# Validate EXP-007: Ridge Meta-Learner Stack — Sanity Check

**🖥 RUNS ON: Kaggle CPU — no GPU needed**

| Item | Value |
|------|-------|
| Target env | Kaggle CPU (no accelerator) |
| Compute | < 1 min |
| Purpose | Verify Ridge stacker logic on synthetic OOF data before GPU run |

Uses **synthetic** 6-model OOF matrix — no competition data, no GPU required.  
Run this first to confirm the stacking and blending logic is correct.

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

print('Kaggle CPU validation — EXP-007 Ridge Stacker')
print('No competition data or GPU required.')

In [ ]:
# ── Ridge stacker (inline — no external import needed) ────────────
class RidgeStacker:
    """Ridge meta-learner: final = blend_alpha*stack + (1-blend_alpha)*heuristic"""
    def __init__(self, alpha=1.0, blend_alpha=0.30, seed=42):
        self.alpha = alpha
        self.blend_alpha = blend_alpha
        self.seed = seed
        self.scaler = StandardScaler()
        self.ridge  = Ridge(alpha=alpha)

    def fit(self, X, y):
        self.ridge.fit(self.scaler.fit_transform(X), y)
        return self

    def predict_stack(self, X):
        return self.ridge.predict(self.scaler.transform(X))

    def blend(self, stack, heuristic):
        return self.blend_alpha * stack + (1 - self.blend_alpha) * heuristic

    def fit_predict_oof(self, X, y, cv=5):
        oof = np.zeros(len(y))
        kf  = KFold(cv, shuffle=True, random_state=self.seed)
        for tr, val in kf.split(X):
            sc = StandardScaler()
            r  = Ridge(alpha=self.alpha)
            r.fit(sc.fit_transform(X[tr]), y[tr])
            oof[val] = r.predict(sc.transform(X[val]))
        self.fit(X, y)
        return oof

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

print('RidgeStacker defined.')

In [ ]:
# ── Synthetic OOF matrix (6 base models: LGB×3 + CB×3) ────────────
rng       = np.random.default_rng(42)
n_samples = 500
n_models  = 6
y_true    = rng.normal(0, 10, n_samples)

oof_matrix = np.zeros((n_samples, n_models))
for i in range(n_models):
    noise_scale = rng.uniform(3, 8)
    bias        = rng.uniform(-2, 2)
    oof_matrix[:, i] = y_true + rng.normal(bias, noise_scale, n_samples)

# Simulate test matrix and model-free heuristic
test_matrix = oof_matrix + rng.normal(0, 1, oof_matrix.shape)
heuristic   = y_true + rng.normal(0, 5, n_samples)

model_names = ['LGB-1', 'LGB-2', 'LGB-3', 'CB-1', 'CB-2', 'CB-3']
print(f'OOF matrix: {oof_matrix.shape}  (n_samples × n_models)')
print('\nIndividual model RMSE:')
for i, name in enumerate(model_names):
    print(f'  {name}: {rmse(y_true, oof_matrix[:, i]):.4f}')

In [ ]:
# ── Compare approaches ────────────────────────────────────────────
# Baseline: simple average
rmse_avg = rmse(y_true, oof_matrix.mean(axis=1))

# Ridge stacker OOF
stacker   = RidgeStacker(alpha=1.0, blend_alpha=0.30, seed=42)
oof_stack = stacker.fit_predict_oof(oof_matrix, y_true, cv=5)
rmse_stack = rmse(y_true, oof_stack)

# Ridge + heuristic blend 0.30/0.70
blend_fixed  = stacker.blend(oof_stack, heuristic)
rmse_blend   = rmse(y_true, blend_fixed)

# Tune blend ratio
best_a, best_r = 0.30, float('inf')
for a in np.arange(0.0, 1.01, 0.05):
    r = rmse(y_true, a * oof_stack + (1 - a) * heuristic)
    if r < best_r: best_r, best_a = r, a

print('=== Ablation ===')
print(f'A. Heuristic only:           {rmse(y_true, heuristic):.4f}')
print(f'B. Simple average (6 models):{rmse_avg:.4f}')
print(f'C. Ridge stack only:         {rmse_stack:.4f}')
print(f'D. Blend 0.30/0.70 (theirs): {rmse_blend:.4f}')
print(f'E. Tuned blend (a={best_a:.2f}):    {best_r:.4f}')

print('\nRidge coef (per model):')
for name, w in zip(model_names, stacker.ridge.coef_):
    print(f'  {name}: {w:+.4f}')

# Test inference
test_stack = stacker.predict_stack(test_matrix)
print(f'\nTest inference: {len(test_stack)} predictions generated OK')

In [ ]:
# ── Assert logic is sound ─────────────────────────────────────────
assert rmse_stack < rmse_avg, 'FAIL: Ridge should beat simple average on synthetic data'
assert len(test_stack) == n_samples, 'FAIL: test inference shape mismatch'
assert len(stacker.ridge.coef_) == n_models, 'FAIL: Ridge should have 1 coef per base model'

print('PASS: all assertions passed — safe to run exp007_ridge_stack__GPU')